## The goal: to scrape the EPA's website and build a clean dataset of superfund sites. 

## The EPA lists the superfund sites on a website and as a PDF.
### So, let's scrape the pdf full of superfund sites! 

In [24]:
from PyPDF2 import PdfReader
from pathlib import Path

ROOT = Path('..')
DATA = ROOT / 'all_datasets' / 'PDFs_of_superfunds'
pdf_path = DATA / "Active_superfund_sites.pdf"

reader = PdfReader(pdf_path)

print("Pages:", len(reader.pages))

text = ""
for page in reader.pages:
    text += page.extract_text() or ""  # Some pages might return None

print(text[:1000])

Pages: 39
NOTICE OF PDF ATTACHMENT 
This PDF file contains an attachment. 
Attachments may be the source file of the PDF or a related file 
(e.g., forms, spreadsheets,  etc). 
To a
ccess the attachment, please download this file to open the 
attachment via PDF reader software (ie. Adobe Acrobat) attachment pane.  
Double-click the attachment to open
:
Example ImageRUN DATE: 03/24/2025 U.S. EPA SUPERFUND PROGRAM
Page 1 of  38
DATA REFRESH DATE: 03/24/2025 12:11:26 2 Source: SEMS Superfund Public User Database
VERSION: 2.00 FOIA-004 All Final NPL Sites
RegionStateSite Name Site ID EPA ID Address City ZipCounty FF IndNAINative American Entity 
(NAI Status)Latitude Longitude NPL Status
Date
01 CT BARKHAMSTED-NEW HARTFORD LANDFILL 0100255 CTD980732333 ROUTE 44 BARKHAMSTED 06063 LITCHFIELD N N +41.893947 -72.989337 10/04/89
01 CT BEACON HEIGHTS LANDFILL 0100180 CTD072122062 BLACKBERRY HILL ROAD BEACON FALLS 06403 NEW HAVEN N N +41.431950 -73.035281 09/08/83
01 CT DURHAM MEADOWS 0100108 CTD00

## Now, let's skip the first page and print out the rest!

### Used AI assistance to help optimize code and avoid overloading the notebook.

In [26]:
for i, page in enumerate(reader.pages[1:], start=2):  
    page_text = page.extract_text() or ""
    print(f"--- Page {i} ---")
    print(page_text[:500])  

--- Page 2 ---
RUN DATE: 03/24/2025 U.S. EPA SUPERFUND PROGRAM
Page 1 of  38
DATA REFRESH DATE: 03/24/2025 12:11:26 2 Source: SEMS Superfund Public User Database
VERSION: 2.00 FOIA-004 All Final NPL Sites
RegionStateSite Name Site ID EPA ID Address City ZipCounty FF IndNAINative American Entity 
(NAI Status)Latitude Longitude NPL Status
Date
01 CT BARKHAMSTED-NEW HARTFORD LANDFILL 0100255 CTD980732333 ROUTE 44 BARKHAMSTED 06063 LITCHFIELD N N +41.893947 -72.989337 10/04/89
01 CT BEACON HEIGHTS LANDFILL 0100180
--- Page 3 ---
RUN DATE: 03/24/2025 U.S. EPA SUPERFUND PROGRAM
Page 2 of  38
DATA REFRESH DATE: 03/24/2025 12:11:26 2 Source: SEMS Superfund Public User Database
VERSION: 2.00 FOIA-004 All Final NPL Sites
RegionStateSite Name Site ID EPA ID Address City ZipCounty FF IndNAINative American Entity 
(NAI Status)Latitude Longitude NPL Status
Date
01 MA WALTON & LONSBURY INC. 0100432 MAD001197755 78 NORTH AVENUE ATTLEBORO 02703 BRISTOL N Y Mashpee Wampanoag Tribe 
(Potential); Wampanoa

In [33]:
pages_text = [page.extract_text() or "" for page in reader.pages[1:]]  # skip first page

all_sites = []

for page_text in pages_text:
    lines = page_text.split("\n")
    for line in lines:
        # skip headers
        if line.startswith("---") or line.startswith("RUN DATE") or line.startswith("Page") or line.startswith("DATA REFRESH"):
            continue

        # Split line by whitespace
        parts = line.split()
        
        # Skip lines that are too short to be data rows
        if len(parts) < 12:
            continue
        
        # First fields are usually Region, State
        region = parts[0]
        state = parts[1]

        # Site ID is the first 6-digit number
        site_id_idx = next((i for i, p in enumerate(parts) if re.match(r"\d{6}", p)), None)
        if site_id_idx is None:
            continue  # skip rows without a site ID
        
        site_name = " ".join(parts[2:site_id_idx])
        site_id = parts[site_id_idx]
        epa_id = parts[site_id_idx + 1]
        
        # The rest: Address, City, Zip, County, Lat, Lon
        # Latitude is the first part that looks like +dd.ddddd
        lat_idx = next((i for i, p in enumerate(parts) if re.match(r"[+-]?\d+\.\d+", p)), None)
        if lat_idx is None:
            continue
        
        address = " ".join(parts[site_id_idx + 2 : lat_idx - 4])
        city = parts[lat_idx - 4]
        zip_code = parts[lat_idx - 3]
        county = parts[lat_idx - 2]
        latitude = float(parts[lat_idx])
        longitude = float(parts[lat_idx + 1])
        
        all_sites.append({
            "Region": region,
            "State": state,
            "Site Name": site_name,
            "Site ID": site_id,
            "EPA ID": epa_id,
            "Address": address,
            "City": city,
            "Zip": zip_code,
            "County": county,
            "Latitude": latitude,
            "Longitude": longitude
        })

# Check first 5 rows
for site in all_sites[:5]:
    print(site)


ValueError: could not convert string to float: 'COROZAL'

In [34]:
import re

pages_text = [page.extract_text() or "" for page in reader.pages[1:]]  # skip first page

all_sites = []

for page_num, page_text in enumerate(pages_text, start=2):
    lines = page_text.split("\n")
    for line in lines:
        # skip headers
        if line.startswith("---") or line.startswith("RUN DATE") or line.startswith("Page"):
            continue
        
        # match lines starting with Region (2 digits) and State (2 letters)
        m = re.match(r"(\d{2})\s+([A-Z]{2})\s+(.+?)\s+\d{6}", line)
        if m:
            region = m.group(1)
            state = m.group(2)
            site_name = m.group(3).strip()
            
            all_sites.append({
                "Region": region,
                "State": state,
                "Site Name": site_name
            })

# check first 5
for site in all_sites[:5]:
    print(site)

{'Region': '01', 'State': 'CT', 'Site Name': 'BARKHAMSTED-NEW HARTFORD LANDFILL'}
{'Region': '01', 'State': 'CT', 'Site Name': 'BEACON HEIGHTS LANDFILL'}
{'Region': '01', 'State': 'CT', 'Site Name': 'DURHAM MEADOWS'}
{'Region': '01', 'State': 'CT', 'Site Name': "GALLUP'S QUARRY"}
{'Region': '01', 'State': 'CT', 'Site Name': 'KELLOGG-DEERING WELL FIELD'}
